# Atención multicabeza

**Capítulo 5 · Universidad de las Hespérides**

Adaptación al español de *Dive into Deep Learning*, Aston Zhang, Zachary C. Lipton, Mu Li y Alexander J. Smola.
Fuente: `locked/chapter_attention-mechanisms-and-transformers/multihead-attention.ipynb` · [Lección original](https://d2l.ai/chapter_attention-mechanisms-and-transformers/multihead-attention.html).
Texto adaptado bajo [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/). [Procedencia y cambios](../PROCEDENCIA.md).
Se conserva la secuencia de las celdas y de los ejercicios; las notas de Hespérides se identifican expresamente.

**Entorno:** ejecuta `uv sync` en la raíz y selecciona su Python como kernel. Las descargas se realizan una vez y quedan en `data/`.
Por defecto, el soporte limita los entrenamientos de `Trainer` a tres épocas y 1024/256 ejemplos para CPU.
Para repetir el régimen completo, inicia Jupyter con `HESPERIDES_COMPLETO=1`. Los ejemplos visuales pequeños conservan su propia configuración explícita.
Los datos de texto en inglés o francés son entradas de los experimentos originales y mantienen su idioma.


In [ ]:
from pathlib import Path
import sys
RAIZ = Path.cwd() if (Path.cwd() / "laboratorio").exists() else Path.cwd().parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
from laboratorio import d2l, configurar, epocas
configurar()


# Atención multicabeza
<a id="sec_multihead-attention"></a>

En la práctica, dado el mismo conjunto de consultas, claves y valores, podemos querer que nuestro modelo combine el conocimiento de diferentes comportamientos del mismo mecanismo de atención, como capturar dependencias de varios rangos (por ejemplo, rango más corto vs. rango más largo) dentro de una secuencia. Así, puede ser beneficioso permitir que nuestro mecanismo de atención utilice conjuntamente diferentes subespacios de representación de consultas, claves y valores.

Con este fin, en lugar de realizar una sola agregación de la atención, las consultas, claves y valores se pueden transformar con proyecciones lineales aprendidas independientemente $h$. Luego estas consultas, claves y valores proyectados $h$ se introducen en la agregación de la atención en paralelo. Al final, las salidas de la agregación de la atención $h$ se concatenan y transforman con otra proyección lineal adquirida para producir la salida final. Este diseño se llama *atención multi-cabeza*, donde cada una de las salidas de la agregación de la atención $h$ es una *cabeza* [Vaswani.Shazeer.Parmar.ea.2017](https://d2l.ai/chapter_references/zreferences.html).
[Referencia fig_multi-head-attention](https://d2l.ai/chapter_attention-mechanisms-and-transformers/multihead-attention.html#fig-multi-head-attention)
describe la atención multi-cabeza.

![Atención multicabeza: las cabezas se concatenan y se transforman linealmente.](../recursos/originales/multi-head-attention.svg)
<a id="fig_multi-head-attention"></a>


In [ ]:
import math
import torch
from torch import nn
from laboratorio import d2l

## Modelo
Antes de proporcionar la implementación de la atención multi-cabeza, formalicemos este modelo matemáticamente. Dada una consulta $\mathbf{q} \in \mathbb{R}^{d_q}$, una llave $\mathbf{k} \in \mathbb{R}^{d_k}$ y un valor $\mathbf{v} \in \mathbb{R}^{d_v}$, cada cabezal de atención $\mathbf{h}_i$ ($i = 1, \ldots, h$) se calcula como

$$\mathbf{h}_i = f(\mathbf W_i^{(q)}\mathbf q, \mathbf W_i^{(k)}\mathbf k,\mathbf W_i^{(v)}\mathbf v) \in \mathbb R^{p_v},$$

donde $\mathbf W_i^{(q)}\in\mathbb R^{p_q\times d_q}$, $\mathbf W_i^{(k)}\in\mathbb R^{p_k\times d_k}$ y $\mathbf W_i^{(v)}\in\mathbb R^{p_v\times d_v}$ son parámetros aprendebles y $f$ es una agrupación de atención, como la atención aditiva y la atención a puntos escalados en [Referencia sec_attention-scoring-functions](https://d2l.ai/chapter_attention-mechanisms-and-transformers/attention-scoring-functions.html#sec-attention-scoring-functions). La salida de atención multi-cabeza es otra transformación lineal a través de parámetros aprendebles $\mathbf W_o\in\mathbb R^{p_o\times h p_v}$ de la concatenación de $h$ cabezas:

$$\mathbf W_o \begin{bmatrix}\mathbf h_1\\\vdots\\\mathbf h_h\end{bmatrix} \in \mathbb{R}^{p_o}.$$

Con base en este diseño, cada cabezal puede atender diferentes partes de la entrada. Funciones más sofisticadas que el promedio ponderado simple se pueden expresar.

## Aplicación
En nuestra implementación, ** elegimos la atención de producto escalar escalado para cada cabezal** de la atención multi-cabeza. Para evitar un crecimiento significativo del costo computacional y el costo de parametrización, establecemos $p_q = p_k = p_v = p_o / h$. Tenga en cuenta que los cabezales $h$ se pueden calcular en paralelo si establecemos el número de salidas de transformaciones lineales para la consulta, clave y valor a $p_q h = p_k h = p_v h = p_o$. En la siguiente implementación, $p_o$ se especifica a través del argumento `num_hiddens`.


In [ ]:
class MultiHeadAttention(d2l.Module):  #@save
    """Atención multi-cabeza."""
    def __init__(self, num_hiddens, num_heads, dropout, bias=False, **kwargs):
        super().__init__()
        self.num_heads = num_heads
        self.attention = d2l.DotProductAttention(dropout)
        self.W_q = nn.LazyLinear(num_hiddens, bias=bias)
        self.W_k = nn.LazyLinear(num_hiddens, bias=bias)
        self.W_v = nn.LazyLinear(num_hiddens, bias=bias)
        self.W_o = nn.LazyLinear(num_hiddens, bias=bias)

    def forward(self, queries, keys, values, valid_lens):
        # Forma de las consultas, claves o valores:
        # (batch_size, no. de consultas o pares de valores clave, num_hiddens)
        # Forma de valid_lens: (batch_size,) o (batch_size, no. de consultas)
        # Después de la transposición, forma de las consultas de salida, claves o valores:
        # (batch_size * num_heads, no. de consultas o pares de valores clave,
        # num_hiddens / num_heads)
        queries = self.transpose_qkv(self.W_q(queries))
        keys = self.transpose_qkv(self.W_k(keys))
        values = self.transpose_qkv(self.W_v(values))

        if valid_lens is not None:
            # En el eje 0, copiar el primer elemento (escalar o vector) para num_heads
            # veces, luego copiar el siguiente elemento, y así sucesivamente
            valid_lens = torch.repeat_interleave(
                valid_lens, repeats=self.num_heads, dim=0)

        # Forma de salida: (batch_size * num_heads, no. de consultas,
        # num_hiddens / num_heads)
        output = self.attention(queries, keys, values, valid_lens)
        # Forma de output_concat: (batch_size, no. de consultas, num_hiddens)
        output_concat = self.transpose_output(output)
        return self.W_o(output_concat)

Para permitir ** cálculo paralelo de múltiples cabezas**, la clase `MultiHeadAttention` anterior utiliza dos métodos de transposición como se define a continuación. Específicamente, el método `transpose_output` invierte el funcionamiento del método `transpose_qkv`.


### Nota docente de Hespérides

Escribe qué información puede ver cada posición. Una máscara causal impide consultar el futuro; una máscara de padding excluye posiciones que no son datos. Comprueba que cada fila de atención suma uno antes de aplicar dropout. Los mapas de atención describen mezclas de valores, pero por sí solos no prueban una explicación causal del modelo.

Vínculo con los apuntes: sesión 5, «Atención multicabeza».


In [ ]:
@d2l.add_to_class(MultiHeadAttention)  #@save
def transpose_qkv(self, X):
    """Transposición para el cálculo paralelo de múltiples cabezas de atención."""
    # Forma de la entrada X: (batch_size, no. de consultas o pares de valores clave,
    # num_hiddens). Forma de salida X: (batch_size, no. de consultas o
    # pares de valores clave, num_heads, num_hiddens / num_heads)
    X = X.reshape(X.shape[0], X.shape[1], self.num_heads, -1)
    # Forma de salida X: (batch_size, num_heads, no. de consultas o valor de clave
    # pares, num_hiddens / num_heads)
    X = X.permute(0, 2, 1, 3)
    # Forma de salida: (batch_size * num_heads, no. de consultas o valor clave
    # pares, num_hiddens / num_heads)
    return X.reshape(-1, X.shape[2], X.shape[3])

@d2l.add_to_class(MultiHeadAttention)  #@save
def transpose_output(self, X):
    """Invertir la operación de transponder_qkv."""
    X = X.reshape(-1, self.num_heads, X.shape[1], X.shape[2])
    X = X.permute(0, 2, 1, 3)
    return X.reshape(X.shape[0], X.shape[1], -1)

Vamos a ** probar nuestra clase implementada** `MultiHeadAttention` usando un ejemplo de juguete donde las claves y los valores son los mismos. Como resultado, la forma de la salida de atención multi-cabeza es (`batch_size`, `num_queries`, `num_hiddens`).


In [ ]:
num_hiddens, num_heads = 100, 5
attention = MultiHeadAttention(num_hiddens, num_heads, 0.5)
batch_size, num_queries, num_kvpairs = 2, 4, 6
valid_lens = torch.tensor([3, 2])
X = torch.ones((batch_size, num_queries, num_hiddens))
Y = torch.ones((batch_size, num_kvpairs, num_hiddens))
d2l.check_shape(attention(X, Y, Y, valid_lens),
                (batch_size, num_queries, num_hiddens))

## Resumen
La atención multi-cabeza combina el conocimiento de la misma concentración de atención a través de diferentes subespacios de representación de consultas, claves y valores. Para calcular múltiples cabezas de atención multi-cabeza en paralelo, se necesita una manipulación de tensor adecuada.

## Ejercicios
1. Visualiza los pesos de atención multicabeza en este experimento.
1. Supongamos que tenemos un modelo entrenado basado en la atención multicabeza y queremos podar las cabezas de atención menos importantes para aumentar la velocidad de predicción. ¿Cómo podemos diseñar experimentos para medir la importancia de un cabezal de atención?


[Debate del original](https://discuss.d2l.ai/t/1635)
